# index-by-tensor — worked example 1: Gather rows by an index tensor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `index-by-tensor`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Indexing a `(N, D)` table with a 1-D `LongTensor` of length `K` selects those rows, producing `(K, D)`. The index tensor's shape becomes the leading shape of the result. This is the per-ray nearest-triangle attribute lookup and the embedding lookup in one move.

## Worked solution

We select `K` rows from a table by a tensor of row indices.

1. `table` is `(N, D)`; `idx` is a 1-D `LongTensor` with values in `[0, N)`.
2. The single expression `table[idx]` performs advanced indexing: for each entry `idx[i]`, it pulls row `idx[i]` of the table.
3. The output shape is `idx.shape + table.shape[1:]` = `(K, D)`. The index shape leads; the trailing feature dimension is carried through.
4. No loop is needed — advanced indexing vectorises the gather. We verify one selected row equals the directly-indexed source row.

In [ ]:
import torch as t

t.manual_seed(0)
table = t.randn(6, 4)
idx = t.tensor([3, 0, 5, 0])

def gather_rows(table, idx):
    return table[idx]

out = gather_rows(table, idx)
print(out.shape)
print('row 0 of out is table[3]:', bool(t.equal(out[0], table[3])))